In [4]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  IN_COLAB = True
  work_dir = '/content/drive/MyDrive/COMP720Project/SampledExplanation'
except:
  IN_COLAB = False
  work_dir = input()

Mounted at /content/drive


In [5]:
# !mkdir /content/drive/MyDrive/COMP720Project/SampledExplanation
# !cp /content/drive/MyDrive/COMP720Project/hbf/explanation_res.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_hbf.csv
# !cp /content/drive/MyDrive/COMP720Project/cf/temp/explanations.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_cf.csv
# !cp /content/drive/MyDrive/COMP720Project/rcbf_all_features/temp/explanations.csv /content/drive/MyDrive/COMP720Project/SampledExplanation/explanations_cbf.csv

In [6]:
import os
os.chdir(work_dir)
os.listdir(work_dir)

['explanations_cf.csv',
 'explanations_cbf.csv',
 'explanations_hbf.csv',
 'summary_means.csv',
 'scored_cbf.csv',
 'score_distributions.csv',
 'scored_cf.csv',
 'scored_hbf.csv']

In [ ]:
import re
import numpy as np
import pandas as pd

SAMPLE_PERS = 100
RANDOM_SEED = 42


In [8]:
FILES = {
    "cbf": ("explanations_cbf.csv", "Content-Based"),
    "cf": ("explanations_cf.csv", "Collaborative"),
    "hbf": ("explanations_hbf.csv", "Hybrid"),
}

In [9]:
################
### Patterns ###
################

PAT_WATCH = re.compile(
    r'Because you watched "(?P<src>.*?)", which (?P<clauses>.*?), '
    r'we think you\'ll like "(?P<rec>.*?)"\.'
)
PAT_NO_HISTORY = re.compile(
    r'"(?P<rec>.*?)" is recommended based on your overall viewing patterns\.'
)
PAT_CF_ITEMS = re.compile(
    r'Because you enjoyed (?P<items>.*?), we think you\'ll like "(?P<rec>.*?)"\.'
)
PAT_CF_GENERIC = re.compile(
    r'"(?P<rec>.*?)" is broadly similar to your recent viewing\.'
)
ITEM_PAT = re.compile(r'(?P<title>.*?) \(your rating: (?P<rating>[\d.]+)\)')

# PAT_WATCH captures the whole "which ... ," middle clause as one blob, since
# the generator (explanations.py) joins its reason clauses with " and " in
# whatever order/subset fired (genre isn't always first, and isn't guaranteed
# to be present at all) rather than a fixed sequence of optional groups.
# Splitting that blob on " and " and matching each piece below is what lets
# parse_cbf_hbf handle any combination.
CLAUSE_GENRE = re.compile(r'shares the (?P<genres>[^()]*) genre\(s\)')
CLAUSE_THEME = "has a similar theme/plot"
CLAUSE_STORY = "touches on similar story elements"
CLAUSE_TASTE = "is often watched by users with similar taste to yours"
CLAUSE_STYLE = "is broadly similar in style"

In [10]:
def parse_cbf_hbf(exp):
    m = PAT_WATCH.match(exp)
    if m:
        src, rec = m.group("src").strip(), m.group("rec").strip()
        clauses = [c.strip() for c in m.group("clauses").split(" and ")]

        genres = []
        theme = story = taste = False
        recognized = 0
        for clause in clauses:
            gm = CLAUSE_GENRE.fullmatch(clause)
            if gm:
                genres = [g.strip() for g in gm.group("genres").split(",") if g.strip()]
                recognized += 1
            elif clause == CLAUSE_THEME:
                theme = True
                recognized += 1
            elif clause == CLAUSE_STORY:
                story = True
                recognized += 1
            elif clause == CLAUSE_TASTE:
                taste = True
                recognized += 1
            # else: unrecognized clause text (e.g. the CLAUSE_STYLE fallback,
            # which only ever appears alone) -- falls through to style_generic below.

        if genres:
            template = "genre"
        elif recognized == 0:
            # No genre and nothing else recognized either -- covers CLAUSE_STYLE
            # ("is broadly similar in style") and any unrecognized clause text.
            template = "style_generic"
        else:
            template = "+".join(name for name, present in
                                 (("theme", theme), ("story", story), ("taste", taste)) if present)

        return {"template": template, "src": src, "rec": rec,
                "n_genres": len(genres), "theme": theme, "story": story, "taste": taste}

    m = PAT_NO_HISTORY.match(exp)
    if m:
        return {"template": "no_history", "src": None, "rec": m.group("rec").strip(),
                "n_genres": 0, "theme": False, "story": False, "taste": False}

    return {"template": "unparsed", "src": None, "rec": None,
            "n_genres": 0, "theme": False, "story": False, "taste": False}



In [11]:
def parse_cf(exp):
    m = PAT_CF_ITEMS.match(exp)
    if m:
        items_str = m.group("items")
        parts = re.split(r"(?<=\)), ", items_str)
        titles, ratings = [], []
        for p in parts:
            im = ITEM_PAT.match(p.strip())
            if im:
                titles.append(im.group("title").strip())
                ratings.append(float(im.group("rating")))
        return {"template": "items", "rec": m.group("rec").strip(), "titles": titles,
                "ratings": ratings, "n_items": len(titles)}
    m = PAT_CF_GENERIC.match(exp)
    if m:
        return {"template": "generic", "rec": m.group("rec").strip(), "titles": [], "ratings": [], "n_items": 0}
    return {"template": "unparsed", "rec": None, "titles": [], "ratings": [], "n_items": 0}




In [12]:

results = {}

for key, (fname, label) in FILES.items():
    df = pd.read_csv(fname)

    rows = []
    for _, row in df.iterrows():
        exp = row["explanation"]
        if key == "cf":
            feat = parse_cf(exp)
            n_evidence = feat["n_items"]
            self_rec = feat["rec"] is not None and feat["rec"].strip().lower() in [t.lower() for t in feat["titles"]]
        else:
            feat = parse_cbf_hbf(exp)
            n_evidence = feat["n_genres"]
            self_rec = (feat["src"] is not None and feat["rec"] is not None
                        and feat["src"].strip().lower() == feat["rec"].strip().lower())

        rows.append({
            "user_id": row["user_id"], "item_id": row["item_id"], "explanation": exp,
            "template": feat["template"], "n_evidence": n_evidence, "self_rec_bug": self_rec,
        })
    parsed = pd.DataFrame(rows)
    parsed.to_csv(f"scored_{key}.csv", index=False)
    results[key] = (label, parsed)

# ---- Parsing report ----
print("=" * 70)
print("Parsed the full explanation file per method (no sampling)")
print("=" * 70)

Parsed the full explanation file per method (no sampling)


In [13]:
summary_rows = []
for key, (label, parsed) in results.items():
    self_rec_rate = parsed["self_rec_bug"].mean()
    template_dist = parsed["template"].value_counts(normalize=True).round(3).to_dict()
    unparsed_count = int((parsed["template"] == "unparsed").sum())
    print(f"\n--- {label} ---")
    print(f"  Unparsed count: {unparsed_count} / {len(parsed)}")
    print(f"  Self-recommendation bug rate: {self_rec_rate:.1%}")
    print(f"  Template mix: {template_dist}")
    summary_rows.append({
        "method": label, "file": key,
        "unparsed_count": unparsed_count,
        "self_rec_bug_rate": round(self_rec_rate, 3),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("summary_means.csv", index=False)
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
display(summary_df)


--- Content-Based ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'genre': 0.974, 'style_generic': 0.019, 'story': 0.007}

--- Collaborative ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'items': 0.553, 'generic': 0.447}

--- Hybrid ---
  Unparsed count: 0 / 1000
  Self-recommendation bug rate: 0.0%
  Template mix: {'genre': 0.945, 'taste': 0.042, 'style_generic': 0.007, 'story': 0.003, 'story+taste': 0.002, 'theme+taste': 0.001}

SUMMARY TABLE


,method,file,unparsed_count,self_rec_bug_rate
0,Content-Based,cbf,0,0.0
1,Collaborative,cf,0,0.0
2,Hybrid,hbf,0,0.0
